# Centauri Bloom 0.1 — Free Colab T4 GPU Training

Train the proposed ~14M `configs/bloom-0.1.json` causal transformer from scratch on a free Google Colab T4 GPU, saving periodic SafeTensors checkpoints to Google Drive.

**Before running:**
1. In Colab, open **Runtime → Change runtime type → T4 GPU** (recommended).
2. Click **Runtime → Run all** (or run cells top-to-bottom).
3. Approve the Google Drive permission prompt so checkpoints survive Colab disconnects.

This notebook uses only reviewed open datasets (`public-domain-english` v1.0.0 and optionally `roneneldan/TinyStories`, `cdla-sharing-1.0`). No tokens, secrets, or automatic uploads are included.

In [ ]:
# 1. Clone Centauri Bloom (main) and install dependencies
import os, sys, subprocess
from pathlib import Path

if Path('pyproject.toml').is_file() and Path('bloom/model.py').is_file():
    REPO = Path.cwd()
elif Path('/content/centauri-bloom/pyproject.toml').is_file():
    REPO = Path('/content/centauri-bloom')
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/InterstellarNerd8760/centauri-bloom.git', '/content/centauri-bloom'], check=True)
    REPO = Path('/content/centauri-bloom')

os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'datasets'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 2. Configure run & mount Google Drive for periodic SafeTensors checkpoints
import json, torch
from pathlib import Path

RUN_NAME = 'bloom-0.1-t4-run1'
TINY_SMOKE_TEST = False        # Set True to test locally on CPU in a few seconds
UPLOAD_TO_HF = True            # Upload final checkpoint to HF if Colab secret HF_TOKEN is set
HF_REPO_ID = 'Centauri-Bloom/Bloom-Text-0.1'
USE_TINYSTORIES = True         # Mix roneneldan/TinyStories (cdla-sharing-1.0) with public-domain-v1
TINYSTORIES_MAX_STORIES = 50_000
TINYSTORIES_VAL_STORIES = 500

STEPS = 3000                   # 3000 * (16 * 2) * 512 = ~49.15M tokens (~25-40 min on T4)
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 3e-4
WARMUP_STEPS = 150
SAVE_EVERY = 500
EVAL_BATCHES = 32
SEED = 42

if TINY_SMOKE_TEST:
    USE_TINYSTORIES = False
    UPLOAD_TO_HF = False
    STEPS = 4
    BATCH_SIZE = 2
    GRAD_ACCUM = 1
    SAVE_EVERY = 2
    EVAL_BATCHES = 2

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not TINY_SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/centauri-bloom-runs') / RUN_NAME
else:
    REPO = globals().get('REPO', Path.cwd())
    OUT_DIR = REPO / 'runs' / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() and not TINY_SMOKE_TEST else 'cpu'
print('Device:', DEVICE, '| Output dir:', OUT_DIR)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Build public-domain-v1, optionally load TinyStories subset, and train 8192-vocab BPE tokenizer
import hashlib, subprocess
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

PD_DIR = REPO / 'runs' / 'public-domain-v1'
if not (PD_DIR / 'manifest.json').is_file():
    subprocess.run([sys.executable, '-m', 'bloom.dataset', 'build', 'datasets/public-domain-v1/sources.json', str(PD_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'bloom.dataset', 'verify', str(PD_DIR)], check=True)

def read_jsonl_texts(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                texts.append(json.loads(line)['text'])
    return texts

train_texts = read_jsonl_texts(PD_DIR / 'data' / 'train-00000.jsonl')
val_texts = read_jsonl_texts(PD_DIR / 'data' / 'validation-00000.jsonl')

if USE_TINYSTORIES:
    from datasets import load_dataset
    ds_train = load_dataset('roneneldan/TinyStories', split=f'train[:{TINYSTORIES_MAX_STORIES}]')
    ds_val = load_dataset('roneneldan/TinyStories', split=f'validation[:{TINYSTORIES_VAL_STORIES}]')
    train_texts.extend(row['text'] for row in ds_train if row['text'].strip())
    val_texts.extend(row['text'] for row in ds_val if row['text'].strip())

print(f'Train docs: {len(train_texts)} | Val docs: {len(val_texts)}')

cfg_raw = json.loads((REPO / 'configs' / 'bloom-0.1.json').read_text())
if TINY_SMOKE_TEST:
    cfg_raw = {'vocab_size': 512, 'context': 32, 'width': 64, 'heads': 4, 'layers': 2}

VOCAB_SIZE = cfg_raw['vocab_size']
CONTEXT = cfg_raw['context']

tok_path = OUT_DIR / 'tokenizer.json'
tokenizer = Tokenizer(BPE(unk_token='<|unk|>'))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=['<|endoftext|>', '<|unk|>'])
tokenizer.train_from_iterator(train_texts, trainer=trainer)
tokenizer.save(str(tok_path))
eot_id = tokenizer.token_to_id('<|endoftext|>')
print('Tokenizer vocab size:', tokenizer.get_vocab_size(), '| saved to', tok_path)

In [ ]:
# 4. Pack non-overlapping token windows and initialize Bloom (~13.99M parameters)
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from bloom.model import Bloom, Config
from bloom.checkpoint import config_hash, save_weights

def encode_windows(texts, tok, context, eot):
    buf = []
    for t in texts:
        buf.extend(tok.encode(t).ids)
        buf.append(eot)
    arr = np.array(buf, dtype=np.int64)
    n_win = (len(arr) - 1) // context
    arr = arr[:n_win * context + 1]
    windows = np.lib.stride_tricks.sliding_window_view(arr, context + 1)[::context]
    x = torch.from_numpy(windows[:, :-1].copy())
    y = torch.from_numpy(windows[:, 1:].copy())
    return TensorDataset(x, y)

train_ds = encode_windows(train_texts, tokenizer, CONTEXT, eot_id)
val_ds = encode_windows(val_texts, tokenizer, CONTEXT, eot_id)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'Train windows: {len(train_ds)} (~{len(train_ds)*CONTEXT:,} tokens) | Val windows: {len(val_ds)}')

config = Config(
    vocab_size=tokenizer.get_vocab_size(),
    context=CONTEXT,
    width=cfg_raw['width'],
    heads=cfg_raw['heads'],
    layers=cfg_raw['layers'],
)
torch.manual_seed(SEED)
model = Bloom(config)
# Scale tied embedding initialization to avoid fp16 logit overflow on step 1
with torch.no_grad():
    torch.nn.init.normal_(model.tokens.weight, mean=0.0, std=0.02)
    torch.nn.init.normal_(model.positions.weight, mean=0.0, std=0.02)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Initialized Bloom: {n_params:,} parameters | config_sha256: {config_hash(config)}')

In [ ]:
# 5. Train with periodic SafeTensors checkpoints saved to Google Drive
import math, time
from itertools import islice
from torch.nn import functional as F

def evaluate(m, loader, device, max_batches):
    m.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in islice(loader, max_batches):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits = m(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction='sum')
            total += loss.item()
            count += y.numel()
    return total / count, count

def save_ckpt(step, val_loss, train_loss, elapsed, tokens_seen):
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    ckpt_file = OUT_DIR / f'step-{step:06d}.safetensors'
    sha = save_weights(ckpt_file, cpu_state, cpu_state)
    latest_file = OUT_DIR / 'latest.safetensors'
    save_weights(latest_file, cpu_state, cpu_state)
    (OUT_DIR / 'config.json').write_text(json.dumps(config.__dict__, indent=2, sort_keys=True) + '\n')
    meta = {
        'step': step,
        'parameters': n_params,
        'config_sha256': config_hash(config),
        'weights_sha256': sha,
        'train_tokens': tokens_seen,
        'last_train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'elapsed_seconds': round(elapsed, 2),
        'use_tinystories': USE_TINYSTORIES,
        'device': DEVICE,
    }
    (OUT_DIR / 'metrics.json').write_text(json.dumps(meta, indent=2, sort_keys=True) + '\n')
    return sha

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

val_before, val_tokens = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Initial val loss: {val_before:.4f} ({val_tokens} tokens)')

it = iter(train_loader)
tokens_seen = 0
t0 = time.perf_counter()
last_loss = val_before

for step in range(1, STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_for_step(step - 1)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(it, None)
        if batch is None:
            it = iter(train_loader)
            batch = next(it)
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()
        tokens_seen += y.numel()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    last_loss = step_loss

    if step == 1 or step % 100 == 0 or step == STEPS:
        dt = time.perf_counter() - t0
        print(f'step {step:4d}/{STEPS} | loss {last_loss:.4f} | {tokens_seen/max(dt,1e-6):,.0f} tok/s | {dt:.1f}s')
    if step % SAVE_EVERY == 0 or step == STEPS:
        vloss, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
        sha = save_ckpt(step, vloss, last_loss, time.perf_counter() - t0, tokens_seen)
        print(f'  -> Saved checkpoint step {step} | val_loss {vloss:.4f} | sha256 {sha[:12]}...')

val_after, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Done in {time.perf_counter() - t0:.1f}s | Val loss: {val_before:.4f} -> {val_after:.4f}')

In [ ]:
# 6. Generate sample completions
def generate(prompt, max_new_tokens=80, temperature=0.8, top_k=40):
    model.eval()
    ids = tokenizer.encode(prompt).ids or [eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = idx[:, -config.context:]
            logits = model(ctx)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < vals[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
    return tokenizer.decode(idx[0].tolist())

for p in ['Once upon a time', 'Alice was', 'The little']:
    print('=== PROMPT:', repr(p), '===')
    print(generate(p))
    print()

In [ ]:
# 7. Optional: upload final checkpoint to Hugging Face (skips cleanly if HF_TOKEN is absent)
import os
from huggingface_hub import HfApi, CommitOperationAdd

hf_token = None
if UPLOAD_TO_HF and not TINY_SMOKE_TEST:
    if IN_COLAB:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = None
    if not hf_token:
        hf_token = os.environ.get('HF_TOKEN')

if not UPLOAD_TO_HF or TINY_SMOKE_TEST:
    print('HF upload skipped (UPLOAD_TO_HF=False or TINY_SMOKE_TEST=True).')
elif not hf_token:
    print('HF upload skipped: no Colab secret or env var named HF_TOKEN found.')
else:
    metrics_obj = json.loads((OUT_DIR / 'metrics.json').read_text())
    readme_text = (
        f"---\nlanguage:\n- en\nlicense: apache-2.0\nlibrary_name: pytorch\ntags:\n"
        f"- text-generation\n- causal-lm\n- from-scratch\n- experimental\n---\n\n"
        f"# Bloom-Text-0.1 (Experimental Run)\n\n"
        f"- **Parameters:** `{metrics_obj['parameters']:,}`\n"
        f"- **Step:** `{metrics_obj['step']}` (`{metrics_obj['train_tokens']:,}` tokens)\n"
        f"- **Validation loss:** `{metrics_obj['val_loss']}`\n"
        f"- **Weights SHA-256:** `{metrics_obj['weights_sha256']}`\n"
        f"- **Status:** Experimental checkpoint; not an official OSAID release.\n"
    )
    (OUT_DIR / 'README.md').write_text(readme_text, encoding='utf-8')
    api = HfApi(token=hf_token)
    ops = [
        CommitOperationAdd(path_in_repo='model.safetensors', path_or_fileobj=str(OUT_DIR / 'latest.safetensors')),
        CommitOperationAdd(path_in_repo='config.json', path_or_fileobj=str(OUT_DIR / 'config.json')),
        CommitOperationAdd(path_in_repo='tokenizer.json', path_or_fileobj=str(OUT_DIR / 'tokenizer.json')),
        CommitOperationAdd(path_in_repo='metrics.json', path_or_fileobj=str(OUT_DIR / 'metrics.json')),
        CommitOperationAdd(path_in_repo='README.md', path_or_fileobj=str(OUT_DIR / 'README.md')),
    ]
    msg = f"Upload {RUN_NAME} step {metrics_obj['step']} (val_loss={metrics_obj['val_loss']})"
    try:
        info = api.create_commit(repo_id=HF_REPO_ID, repo_type='model', operations=ops, commit_message=msg)
        print('Uploaded directly to HF main:', info.commit_url)
    except Exception as exc:
        if '403' in str(exc) or 'create_pr' in str(exc):
            info = api.create_commit(repo_id=HF_REPO_ID, repo_type='model', operations=ops, commit_message=msg, create_pr=True)
            print('Opened HF PR (token lacks direct write):', info.pr_url)
        else:
            raise